# Лабораторная 1: выборочные характеристики и распределения

В этой работе строятся 8 выборок из 4 распределений (по 2 объема: 100 и 1000), считаются выборочные характеристики, результаты сравниваются с `numpy`, а затем визуально сравниваются гистограммы и теоретические плотности/функции вероятности.


In [2]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st


ModuleNotFoundError: No module named 'matplotlib'

## Генерация выборок и как работает `rvs(...)`

Для генерации используется метод `scipy.stats.<distribution>.rvs(...)`:
- `size` задает объем выборки;
- параметры распределения (`loc`, `scale`, `p`, `n`) задают закон распределения;
- `random_state` фиксирует генератор случайных чисел и делает результат воспроизводимым.

В этой работе зафиксировано `GROUP_NUMBER = 12`, поэтому все результаты можно повторить на любой машине при тех же параметрах.


In [ ]:
random_state = 12

## Математическое обоснование для выборок

### 1) Равномерное распределение `U(2, 7)` (выборки №1 и №2)
Если `X ~ U(a, b)`, то:
\[
\mathbb{E}[X] = \frac{a+b}{2}, \quad
\mathbb{D}[X] = \frac{(b-a)^2}{12}, \quad
\sigma = \sqrt{\mathbb{D}[X]}.
\]
Для `a=2, b=7`: \(\mathbb{E}[X]=4.5\), \(\mathbb{D}[X]=\frac{25}{12}\approx 2.0833\), \(\sigma\approx1.4434\).

### 2) Распределение Бернулли `B(0.27)` (выборки №3 и №4)
Если `X ~ Bernoulli(p)`, то:
\[
\mathbb{P}(X=1)=p,\; \mathbb{P}(X=0)=1-p,
\]
\[
\mathbb{E}[X]=p, \quad
\mathbb{D}[X]=p(1-p), \quad
\sigma=\sqrt{p(1-p)}.
\]
Для `p=0.27`: \(\mathbb{E}[X]=0.27\), \(\mathbb{D}[X]=0.1971\), \(\sigma\approx0.4440\).

### 3) Биномиальное распределение `Bin(n=12, p=0.35)` (выборки №5 и №6)
Если `X ~ Bin(n, p)`, то:
\[
\mathbb{P}(X=k)=\binom{n}{k}p^k(1-p)^{n-k}, \quad k=0,1,\dots,n,
\]
\[
\mathbb{E}[X]=np, \quad
\mathbb{D}[X]=np(1-p), \quad
\sigma=\sqrt{np(1-p)}.
\]
Для `n=12, p=0.35`: \(\mathbb{E}[X]=4.2\), \(\mathbb{D}[X]=2.73\), \(\sigma\approx1.6523\).

### 4) Нормальное распределение `N(4.5, 1.8^2)` (выборки №7 и №8)
Если `X ~ N(\mu, \sigma^2)`, то:
\[
\mathbb{E}[X]=\mu, \quad \mathbb{D}[X]=\sigma^2.
\]
Для `\mu=4.5, \sigma=1.8`: \(\mathbb{E}[X]=4.5\), \(\mathbb{D}[X]=3.24\), \(\sigma=1.8\).

При увеличении размера выборки с 100 до 1000 оценки обычно становятся ближе к теоретическим значениям (закон больших чисел).


In [ ]:
samples_dict = {}

samples_dict.update({
        "uniform_100": st.uniform.rvs(loc=2, scale=5, size=100, random_state=random_state),
        "uniform_1000": st.uniform.rvs(loc=2, scale=5, size=1000, random_state=random_state),
    })
samples_dict.update({
        "bernoulli_100": st.bernoulli.rvs(p=0.27, size=100, random_state=random_state),
        "bernoulli_1000": st.bernoulli.rvs(p=0.27, size=1000, random_state=random_state),
    })
samples_dict.update( {
        "binom_100": st.binom.rvs(n=12, p=0.35, size=100, random_state=random_state),
        "binom_1000": st.binom.rvs(n=12, p=0.35, size=1000, random_state=random_state),
    }

)
samples_dict.update({
        "norm_100": st.norm.rvs(loc=4.5, scale=1.8, size=100, random_state=random_state),
        "norm_1000": st.norm.rvs(loc=4.5, scale=1.8, size=1000, random_state=random_state),
    }

)


## Собственные функции статистик (без `numpy`)

Используются формулы:
\[
\bar{x}=\frac{1}{n}\sum_{i=1}^n x_i, \quad
D=\frac{1}{n}\sum_{i=1}^n(x_i-\bar{x})^2, \quad
s=\sqrt{D}.
\]


In [ ]:
def mean(X):
    total = 0.0
    count = 0
    for xi in X:
        total += float(xi)
        count += 1
    return total / count if count else float("nan")


def var(X):
    count = len(X)
    if count == 0:
        return float("nan")

    Xr = mean(X)
    sq_sum = 0.0
    for value in X:
        sq_sum +=  (float(value) - Xr)**2
    return sq_sum / count


def std(X):
    return math.sqrt(var(X))


## Таблица результатов (`pandas`)

Шапка таблицы соответствует заданию:
- № выборки
- среднее своё
- дисперсия своя
- стандартное отклонение своё
- среднее numpy
- дисперсия numpy
- стандартное отклонение numpy


In [ ]:
rows = []
for item in samples:
    arr = item["sample"]
    rows.append({
        "№ выборки": item["sample_no"],
        "среднее своё": my_mean(arr),
        "дисперсия своя": my_variance(arr),
        "стандартное отклонение своё": my_std(arr),
        "среднее numpy": float(np.mean(arr)),
        "дисперсия numpy": float(np.var(arr)),
        "стандартное отклонение numpy": float(np.std(arr)),
    })

results_df = pd.DataFrame(rows).sort_values("№ выборки")
results_df.iloc[:, 1:] = results_df.iloc[:, 1:].round(6)
results_df


## Гистограммы и теоретические кривые (`matplotlib`)

Для каждой из 8 выборок строится один график:
- гистограмма плотностей относительных частот (`density=True`);
- теоретическая плотность вероятности (`pdf`) для непрерывных распределений;
- теоретическая функция вероятности (`pmf`) для дискретных распределений.


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()

for ax, item in zip(axes, samples):
    arr = item["sample"]
    dist = item["dist"]
    params = item["params"]

    if item["is_discrete"]:
        x_min = int(np.min(arr))
        x_max = int(np.max(arr))
        bins = np.arange(x_min - 0.5, x_max + 1.5, 1)
        x = np.arange(x_min, x_max + 1)

        ax.hist(
            arr,
            bins=bins,
            density=True,
            alpha=0.65,
            color="#5FA8D3",
            edgecolor="black",
            label="Гистограмма относительных частот",
        )
        y = dist.pmf(x, **params)
        ax.plot(x, y, "o-", color="#D1495B", linewidth=2, label="Функция вероятности")
    else:
        ax.hist(
            arr,
            bins=20,
            density=True,
            alpha=0.65,
            color="#5FA8D3",
            edgecolor="black",
            label="Гистограмма относительных частот",
        )
        x = np.linspace(np.min(arr), np.max(arr), 400)
        y = dist.pdf(x, **params)
        ax.plot(x, y, color="#D1495B", linewidth=2, label="Плотность вероятности")

    ax.set_title(f"Выборка {item['sample_no']}: {item['dist_title']}, n={item['size']}")
    ax.set_xlabel("x")
    ax.set_ylabel("Плотность / вероятность")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()
